# Расчет различных критериев

In [87]:
import osmnx as ox
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import os

from pathlib import Path

from datetime import time


ModuleNotFoundError: No module named 'openpyxl'

In [61]:
city_crs = 28409
SCENARIO = 'scen_0'
ROUTES_PATH = fr'X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\4_result\{SCENARIO}\routes.gpkg' # МАРШРУТЫ

routes = gpd.read_file(os.path.join(ROUTES_PATH)).to_crs(city_crs) # Маршруты
gdf_routes = routes.query("main_route == 'true' or main_route.isnull()").copy()
gdf_routes['interval_min_morning06-10'] = gdf_routes['interval_min_morning06-10'].astype(float)
gdf_routes.head()

,ya_line_id,ya_route_id,ya_vhc_type,ya_route_name,route_name_with_vhc_prefix,file_name,start_stop_id,end_stop_id,start_stop_name,end_stop_name,route_description,route_name_with_vhc_prefix_routeid_description,main_route,interval_min_morning06-10,capacity_vehicles_at_route,length,krivo,obosoblen,intermodalnost,dtp,2_timework,3_class,8_uds,9_anchor,10_popul,2_start_time,routes_wuds_b_percent_mean,geometry
0,2161483905,2161483940,bus,11,bus_11,11,stop__9884354,stop__10061013,Ижевск – Собор Александра Невского,Улица Фурманова,,"11_[""2161483940"";""""]",true,14.0,None,6.92,1.11,0.00,35.0,0.00,23,3,32,2,2,5,31.9,"LINESTRING (9634444.857 6304730.034, 9634418.7..."
1,2161483905,2161483946,bus,11,bus_11,11,stop__9884455,stop__9884352,Улица Фурманова,Ижевск – Собор Александра Невского,,"11_[""2161483946"";""""]",true,14.0,None,7.09,1.14,0.00,35.0,0.00,23,3,32,2,2,5,31.9,"LINESTRING (9631958.022 6299080.08, 9631958.96..."
2,2161483924,2161483952,bus,12,bus_12,12,stop__9884357,stop__9884310,Станкострой,Автозавод - Разворотное кольцо,,"12_[""2161483952"";""""]",true,20.0,None,12.89,1.45,19.89,35.7,3.22,21,3,31,2,14,4,31.0,"LINESTRING (9633034.046 6304670.72, 9633036.20..."
3,2161483924,2161483955,bus,12,bus_12,12,stop__9884309,stop__9884357,Автозавод – Разворотное кольцо,Станкострой,,"12_[""2161483955"";""""]",true,20.0,None,13.08,1.47,19.57,35.7,3.27,21,3,31,2,14,4,31.0,"LINESTRING (9640668.133 6309216.425, 9640637.0..."
4,2161483926,2161484020,bus,15,bus_15,15,stop__9884471,stop__9884357,Металлобаза,Станкострой,,"15_[""2161484020"";""""]",true,120.0,None,8.19,1.30,0.00,26.5,0.00,15,3,57,2,4,5,57.0,"LINESTRING (9639115.503 6303107.303, 9639116.1..."


## Обработка слоя с маршрутами

In [62]:
gdf_routes.columns

Index(['ya_line_id', 'ya_route_id', 'ya_vhc_type', 'ya_route_name',
       'route_name_with_vhc_prefix', 'file_name', 'start_stop_id',
       'end_stop_id', 'start_stop_name', 'end_stop_name', 'route_description',
       'route_name_with_vhc_prefix_routeid_description', 'main_route',
       'interval_min_morning06-10', 'capacity_vehicles_at_route', 'length',
       'krivo', 'obosoblen', 'intermodalnost', 'dtp', '2_timework', '3_class',
       '8_uds', '9_anchor', '10_popul', '2_start_time',
       'routes_wuds_b_percent_mean', 'geometry'],
      dtype='str')

In [63]:
gdf_grade = gdf_routes[['ya_line_id','ya_route_id','ya_vhc_type', 'ya_route_name', 'start_stop_name', 'end_stop_name',  'length',
       'interval_min_morning06-10', '2_start_time', '2_timework',  '3_class', 'intermodalnost',  'krivo', 'obosoblen', 'dtp', 
       '8_uds', '9_anchor', '10_popul', 'geometry']].reset_index()

In [64]:
rename_col = {"interval_min_morning06-10": "N1", "2_start_time": "N2_1", "2_timework": "N2", "3_class": "N3",
             "intermodalnost": "N4", "krivo": "N5", "obosoblen": "U1", "dtp": "U2", "8_uds": "U3",
              "9_anchor": "D1",  "10_popul": "D2"
             }

In [65]:
gdf_grade = gdf_grade.rename(columns=rename_col).groupby('ya_line_id').first().reset_index()

In [66]:
pd.set_option('display.max_columns', None)
gdf_grade.head()

,ya_line_id,index,ya_route_id,ya_vhc_type,ya_route_name,start_stop_name,end_stop_name,length,N1,N2_1,N2,N3,N4,N5,U1,U2,U3,D1,D2,geometry
0,2161483899,57,2161484033,tramway,1,Сквер Металлургов,Московская улица,11.69,11.0,4,23,3,100.0,1.31,100.00,5.85,0,2,16,"LINESTRING (9633289.013 6308266.041, 9633302.6..."
1,2161483900,78,2161484031,trolleybus,1,Дом печати / Троллейбусное кольцо,Центр,10.05,25.0,5,23,3,19.5,1.29,10.29,5.03,25,2,16,"LINESTRING (9639552.416 6310726.721, 9639518.4..."
2,2161483901,59,2161483920,tramway,10,Улица Ворошилова,Сквер Металлургов,12.78,8.5,5,23,3,100.0,2.22,99.00,6.39,0,2,21,"LINESTRING (9639052.117 6308093.144, 9639014.2..."
3,2161483903,80,2161483927,trolleybus,10,Улица Ворошилова,Посёлок Машиностроителей,17.78,15.0,5,23,3,19.7,1.57,0.00,3.56,25,1,21,"LINESTRING (9639132.959 6308364.645, 9639140.7..."
4,2161483904,61,2161483936,tramway,11,Промышленная улица,Улица Ворошилова,7.10,27.0,5,18,3,100.0,1.52,99.00,0.00,0,1,9,"LINESTRING (9637369.954 6303909.689, 9637365.6..."


In [67]:
gdf_grade.dtypes

ya_line_id              str
index                 int64
ya_route_id             str
ya_vhc_type             str
ya_route_name           str
start_stop_name         str
end_stop_name           str
length              float64
N1                  float64
N2_1                  int32
N2                    int32
N3                    int32
N4                  float64
N5                  float64
U1                  float64
U2                  float64
U3                    int32
D1                    int32
D2                    int32
geometry           geometry
dtype: object

## Веса

### Веса, забираемые из папки

In [68]:
w_folder = Path(r"X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\1_data\weights")
w_files = [f.name for f in w_folder.iterdir() if f.is_file()]
w_files[-1]
w_file_path = w_folder / w_files[-1]
weigths = pd.read_csv(w_file_path).iloc[:-1]

# weigths['median'] = weigths.iloc[:, 1:6].median(axis=1)
weigths

,Полный критерий,Краткий критерий,ЛЛМ,ЕП,УС,МЖ,ИР,ЛИ
0,Критерий интервала маршрута НГПТ (Н1),N1,20,5,8,18,14,10
1,Критерий времени работы маршрутов НГПТ (Н2),N2,10,5,4,8,14,10
2,Критерий класса подвижного состава НГПТ (Н3),N3,9,5,10,12,14,10
3,Критерий интермодальности маршрута НГПТ (Н4),N4,13,15,10,6,7,10
4,Критерий криволинейности маршрута НГПТ (Н5),N5,6,5,4,8,14,5
5,Критерий обособленности маршрута НГПТ (У1),U1,13,10,10,14,14,15
6,Критерий наличия мест концентрации ДТП на марш...,U2,3,5,4,4,4,5
7,Критерий наличия неподходящих участков УДС на ...,U3,4,10,10,4,5,5
8,Критерий доступности маршрута НГПТ для населен...,D1,15,20,20,20,7,15
9,Критерий наличия якорных объектов притяжения в...,D2,7,20,20,6,7,15


### Расчет весов

In [69]:
from scipy import stats

def trimmed_mean_per_criterion(df, trim=0.1, axis=1):
    """
    Вычисляет усечённое среднее для каждого критерия (строки).
    
    Параметры
    ----------
    df : pandas.DataFrame
        Датафрейм: строки — критерии, столбцы — эксперты, значения — веса.
    trim : float, default=0.1
        Доля отбрасываемых наименьших и наибольших значений с каждого конца.
    axis : int, default=1
        Ось, вдоль которой вычисляется статистика (1 — по строкам).
    
    Возвращает
    ----------
    pandas.Series
        Серия с усечёнными средними для каждого критерия.
    """
    # Выбираем только числовые столбцы (эксперты)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df_numeric = df[numeric_cols]
    
    if df_numeric.empty:
        raise ValueError("Нет числовых данных!")
    
    # Применяем trim_mean к каждой строке
    trimmed_means = df_numeric.apply(
        lambda row: stats.trim_mean(row.dropna(), proportiontocut=trim),
        axis=axis
    )
    trimmed_means.name = f'trimmed_mean_{trim}'
    return trimmed_means

def median_per_criterion(df, axis=1):
    """
    Вычисляет медиану для каждого критерия (строки).
    
    Параметры
    ----------
    df : pandas.DataFrame
        Датафрейм: строки — критерии, столбцы — эксперты.
    axis : int, default=1
        Ось вычисления медианы (1 — по строкам).
    
    Возвращает
    ----------
    pandas.Series
        Серия с медианными значениями для каждого критерия.
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df_numeric = df[numeric_cols]
    
    if df_numeric.empty:
        raise ValueError("Нет числовых данных!")
    
    medians = df_numeric.median(axis=axis, skipna=True)
    medians.name = 'median'
    return medians

In [70]:
my_median = trimmed_mean_per_criterion(weigths)

weights_df = pd.DataFrame({
    'feature': weigths['Краткий критерий'],
    'weight_m': median_per_criterion(weigths).values / 100,
    'weight_u': trimmed_mean_per_criterion(weigths, trim=0.2).values / 100
}).set_index('feature')

weights_df

,weight_m,weight_u
feature,,
N1,0.120,0.1250
N2,0.090,0.0825
N3,0.100,0.1025
N4,0.100,0.1000
N5,0.055,0.0600
U1,0.135,0.1275
U2,0.040,0.0425
U3,0.050,0.0600
D1,0.175,0.1750


## Критерии

## Итоговый подсчет

### Подсчет итогового балла на основе весов

In [71]:
# %autoreload 2
import krit

In [72]:
SCORING_FUNCTIONS: dict[str, callable] = {
    'N1': krit.calculate_N1_points,
    'N2': krit.calculate_N2_points,
    'N3': krit.calculate_N3_points,
    'N4': krit.calculate_N4_points,
    'N5': krit.calculate_N5_points,
    'U1': krit.calculate_U1_points,
    'U2': krit.calculate_U2_points,
    'U3': krit.calculate_U3_points,
    'D1': krit.calculate_D1_points,
    'D2': krit.calculate_D2_points,
}

In [73]:
# Применяем все функции
df_scores = pd.DataFrame(index=gdf_grade.index)

for col in gdf_grade.columns:
    if col in SCORING_FUNCTIONS:
        df_scores[col] = gdf_grade[col].apply(SCORING_FUNCTIONS[col])


# Валидация результатов
assert df_scores.min().min() >= 0, "Есть оценки ниже минимального!"
assert df_scores.max().max() <= 2, "Есть оценки выше максимального!"
print("✅ Все оценки в диапазоне [0, 2]")
df_scores['total_score'] = df_scores.sum(axis=1)

✅ Все оценки в диапазоне [0, 2]


In [89]:
# Приводим веса к Series для удобного умножения
weights_series = weights_df['weight_m']*10

# # Умножаем каждый столбец на соответствующий вес
df_weighted = df_scores.mul(weights_series, axis=1)

# # Добавляем итоговый балл по строке
df_weighted['total_score'] = df_weighted.sum(axis=1).round(2)

print(f"Мин/макс итогового балла: {df_weighted['total_score'].min():.2f} / {df_weighted['total_score'].max():.2f}")
df_weighted.sort_values('total_score',ascending=False).head(15)

Мин/макс итогового балла: 4.40 / 16.20


,D1,D2,N1,N2,N3,N4,N5,U1,U2,U3,total_score
2,3.50,2.20,1.2,1.8,1.0,2.0,0.00,2.7,0.8,1.0,16.20
0,3.50,1.65,0.0,1.8,1.0,2.0,1.10,2.7,0.8,1.0,15.55
32,3.50,1.10,1.2,0.0,1.0,2.0,0.55,2.7,0.8,1.0,13.85
6,3.50,1.65,0.0,0.0,1.0,2.0,1.10,2.7,0.8,1.0,13.75
21,3.50,1.65,0.0,0.9,1.0,2.0,0.55,2.7,0.4,1.0,13.70
13,3.50,1.65,1.2,1.8,1.0,2.0,1.10,0.0,0.8,0.5,13.55
12,3.50,1.65,0.0,0.0,1.0,2.0,1.10,2.7,0.4,1.0,13.35
25,3.50,1.65,0.0,0.0,1.0,2.0,1.10,2.7,0.4,1.0,13.35
26,3.50,1.65,1.2,1.8,1.0,2.0,0.55,0.0,0.8,0.5,13.00
30,3.50,1.10,0.0,0.0,1.0,2.0,0.55,2.7,0.4,1.0,12.25


In [90]:
# Объединение по индексу (горизонтально)
result = pd.concat([gdf_grade, df_weighted['total_score']], axis=1)

In [91]:
result.sort_values('total_score',ascending=False).head(10)

,ya_line_id,index,ya_route_id,ya_vhc_type,ya_route_name,start_stop_name,end_stop_name,length,N1,N2_1,N2,N3,N4,N5,U1,U2,U3,D1,D2,geometry,total_score
2,2161483901,59,2161483920,tramway,10,Улица Ворошилова,Сквер Металлургов,12.78,8.500000,5,23,3,100.0,2.22,99.0,6.39,0,2,21,"LINESTRING (9639052.117 6308093.144, 9639014.2...",16.20
0,2161483899,57,2161484033,tramway,1,Сквер Металлургов,Московская улица,11.69,11.000000,4,23,3,100.0,1.31,100.0,5.85,0,2,16,"LINESTRING (9633289.013 6308266.041, 9633302.6...",15.55
32,2161483966,70,2161484080,tramway,5,Буммаш,Огнеупорная улица,14.67,8.000000,5,19,3,100.0,1.56,100.0,7.34,0,2,14,"LINESTRING (9637267.711 6309147.497, 9637267.5...",13.85
6,2161483906,63,2161483949,tramway,12,Улица Ворошилова,Московская улица,13.35,14.117647,5,17,3,100.0,1.30,100.0,6.67,0,2,17,"LINESTRING (9639052.117 6308093.144, 9639014.2...",13.75
21,2161483932,65,2161484098,tramway,2,Буммаш,Промышленная улица,10.05,12.631579,4,22,3,100.0,1.95,100.0,3.35,0,2,18,"LINESTRING (9637267.711 6309147.497, 9637267.5...",13.70
13,2161483917,32,2179995472,bus,40,Ижевск – Центр,Завод пластмасс,11.85,7.500000,4,23,3,37.5,1.45,0.0,0.00,29,2,16,"LINESTRING (9634565.539 6304720.436, 9634581.7...",13.55
12,2161483915,68,2161484012,tramway,4,Промышленная улица,Сквер Металлургов,8.86,34.285714,6,18,3,100.0,1.48,100.0,4.43,0,2,19,"LINESTRING (9637369.954 6303909.689, 9637365.6...",13.35
25,2161483950,76,2161484057,tramway,9,Буммаш,Московская улица,13.02,13.000000,4,17,3,100.0,1.25,100.0,4.34,0,2,15,"LINESTRING (9637267.711 6309147.497, 9637267.5...",13.35
26,2161483951,91,2161484058,trolleybus,9,Посёлок Машиностроителей,Городок Металлургов,26.73,9.000000,5,24,3,47.8,1.80,0.0,13.36,31,2,18,"LINESTRING (9628966.413 6303341.049, 9628995.3...",13.00
30,2161483959,67,3079949411,tramway,3,Промышленная улица,Московская улица,9.42,480.000000,0,18,3,100.0,1.60,100.0,4.71,0,2,11,"LINESTRING (9637369.954 6303909.689, 9637365.6...",12.25


In [95]:
# result[['ya_vhc_type', 'ya_route_name','total_score','rank']].sort_values('total_score',ascending=False).to_csv('res.csv')

In [96]:
result['rank'] = result['total_score'].apply(
    lambda x: 1 if x >= 14 else (2 if x >= 10 else 3)
)

In [97]:
counts = result['rank'].value_counts(sort=False)
counts

rank
1     2
2    16
3    32
Name: count, dtype: int64

In [59]:
result = result.set_crs(epsg='28409')
result.to_file(fr'X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\4_result\{SCENARIO}\routes_kriterii.gpkg', driver='GPKG')